# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the FAIR² protein dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
# Print basic metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# Discover record sets and fields with their @id
print('Record sets:')
record_sets = []
for rset in metadata.record_sets:
    print(f"  @id: {rset['@id']}, name: {rset['name']}")
    record_sets.append(rset['@id'])

print('\nFields per record set:')
record_set_fields = {}
for rset in metadata.record_sets:
    print(f"  Record set @id: {rset['@id']} ({rset['name']})")
    field_ids = []
    for field in rset['fields']:
        fid = field['@id']
        print(f"    Field @id: {fid}, name: {field.get('name','')}, dataType: {field.get('dataType','')}")
        field_ids.append(fid)
    record_set_fields[rset['@id']] = field_ids

if not record_sets:
    print('No record sets found in metadata.')

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# If available, extract data from each record set
# Choose one default record set if available
if record_sets:
    rs_to_use = record_sets[0]
    print(f"Extracting records for record set: {rs_to_use}")
    dataframes = {}
    for rs in record_sets:
        print(f"  Loading records for {rs} ...")
        try:
            records = list(dataset.records(record_set=rs))
            dataframes[rs] = pd.DataFrame(records)
            print(f"    Columns: {dataframes[rs].columns.tolist()}")
        except Exception as e:
            print(f"    Failed to extract: {e}")

    print("\nFirst 5 records from first record set:")
    display(dataframes[rs_to_use].head())
else:
    print('No record sets found in metadata.')

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# Perform EDA on the first record set if it is available and has numeric fields
import numpy as np
import warnings

if record_sets:
    df = dataframes[rs_to_use]
    print(f"Sample columns in this record set: {df.columns.tolist()}")

    # Try to find a numeric field
    numeric_field = None
    for col in df.columns:
        # If the dtype is not object, it's likely numeric
        if np.issubdtype(df[col].dropna().dtype, np.number):
            numeric_field = col
            break
        # Sometimes it's a string of numbers; try to convert
        try:
            df[col] = pd.to_numeric(df[col], errors='ignore')
            if np.issubdtype(df[col].dropna().dtype, np.number):
                numeric_field = col
                break
        except Exception:
            continue

    if numeric_field is not None:
        print(f"Using field '{numeric_field}' for numeric analysis.")

        threshold = df[numeric_field].mean() if df[numeric_field].dtype.kind in 'fi' else 0
        if not np.issubdtype(df[numeric_field].dtype, np.number):
            # Try to forcibly convert
            with warnings.catch_warnings():
                warnings.simplefilter('ignore')
                df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
        display(filtered_df.head())

        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / (filtered_df[numeric_field].std() + 1e-10)
        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Try to find a group/categorical field
        group_field = None
        for col in df.columns:
            if df[col].nunique() > 1 and df[col].nunique() < 8 and col != numeric_field:
                group_field = col
                break
        if group_field:
            print(f"Grouping by '{group_field}'")
            grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
            print(f"Grouped data by {group_field}:")
            display(grouped_df.head())
        else:
            print("No suitable group field found for grouping.")
    else:
        print("No numeric field available for analysis in this record set.")
else:
    print('No record sets or dataframes were loaded for EDA.')

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if record_sets and numeric_field is not None and df.shape[0] > 0:
    plt.figure(figsize=(7,4))
    sns.histplot(df[numeric_field].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.show()

    # Pair plot for numeric fields if more than one
    numeric_df = df.select_dtypes(include=np.number)
    if numeric_df.shape[1] > 1:
        sns.pairplot(numeric_df)
        plt.suptitle("Pairplot of numeric fields")
        plt.show()
else:
    print("No suitable numeric data for visualization.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.